# 🚀 StoryDiffusion x Agentic Comic Generator
Notebook này kết hợp **LangGraph Agents** (sáng tạo nội dung, kịch bản) và **StoryDiffusion** (giữ nhất quán khuôn mặt 100% bằng Consistent Self-Attention) để tự động sinh truyện tranh.

> **Lưu ý:** Vui lòng bật GPU (T4 hoặc A100) trước khi chạy.

In [ ]:
!pip install -qU diffusers transformers accelerate langchain langchain-openai langgraph pydantic
print('✅ Dependencies Installed!')

In [ ]:
import os
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# ==========================================
# 🔑 ĐIỀN OPENAI API KEY CỦA BẠN VÀO ĐÂY
os.environ['OPENAI_API_KEY'] = 'sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'
# ==========================================

llm_json = ChatOpenAI(model='gpt-4o', temperature=0.7, model_kwargs={'response_format': {'type': 'json_object'}})

def generate_comic_script(idea, num_panels=6):
    prompt = ChatPromptTemplate.from_messages([
        ('system', f'''You are an expert comic storyboarder.
Create a comic script based on the user\'s idea.
Your output MUST be a valid JSON with two fields:
1. "character_prompt": A detailed visual description of the main character (e.g. "a young man, black hair, wearing a blue suit").
2. "panels": An array of EXACTLY {num_panels} strings. Each string is an image generation prompt for a comic panel.
CRITICAL: The character description MUST be prepended to EACH panel prompt to ensure consistency.
Example:
{{
  "character_prompt": "a young man, black hair, wearing a blue suit",
  "panels": [
      "a young man, black hair, wearing a blue suit, waking up in bed, morning sunlight",
      "a young man, black hair, wearing a blue suit, eating breakfast, kitchen"
  ]
}}'''),
        ('human', 'Idea: {idea}')
    ])
    
    chain = prompt | llm_json
    print('🧠 Agent đang sáng tác kịch bản...')
    res = chain.invoke({'idea': idea})
    return json.loads(res.content)

# Test Agent
idea = 'Cuộc sống hàng ngày của một nữ phi hành gia trẻ tuổi trên trạm vũ trụ ISS'
script = generate_comic_script(idea)
print(json.dumps(script, indent=2, ensure_ascii=False))

In [ ]:
import torch
import copy
from diffusers import StableDiffusionXLPipeline, DDIMScheduler
from sd_utils.gradio_utils import AttnProcessor2_0 as AttnProcessor
from sd_utils.gradio_utils import cal_attn_mask_xl
from sd_utils.gradio_utils import SpatialAttnProcessor2_0

device = 'cuda' if torch.cuda.is_available() else 'cpu'
sd_model_path = 'SG161222/RealVisXL_V4.0' # SDXL base model

print('⏳ Loading SDXL Pipeline...')
pipe = StableDiffusionXLPipeline.from_pretrained(sd_model_path, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to(device)
pipe.enable_freeu(s1=0.6, s2=0.4, b1=1.1, b2=1.2)
pipe.scheduler = DDIMScheduler.from_config(pipe.scheduler.config)
pipe.scheduler.set_timesteps(50)

def inject_story_diffusion(pipe, num_prompts, id_length=3, sa32=0.5, sa64=0.5, height=768, width=768):
    unet = pipe.unet
    attn_procs = {}
    total_length = num_prompts

    for name in unet.attn_processors.keys():
        cross_attention_dim = None if name.endswith('attn1.processor') else unet.config.cross_attention_dim
        if name.startswith('mid_block'):
            hidden_size = unet.config.block_out_channels[-1]
        elif name.startswith('up_blocks'):
            block_id = int(name[len('up_blocks.')])
            hidden_size = list(reversed(unet.config.block_out_channels))[block_id]
        elif name.startswith('down_blocks'):
            block_id = int(name[len('down_blocks.')])
            hidden_size = unet.config.block_out_channels[block_id]

        if cross_attention_dim is None and name.startswith('up_blocks'):
            attn_procs[name] = SpatialAttnProcessor2_0(id_length=id_length)
        else:
            attn_procs[name] = AttnProcessor()

    unet.set_attn_processor(copy.deepcopy(attn_procs))
    print('✅ StoryDiffusion Spatial Attention Injected!')

    # Note: cal_attn_mask_xl sets global variables in sd_utils.gradio_utils
    import sd_utils.gradio_utils as gradio_utils
    gradio_utils.mask1024, gradio_utils.mask4096 = cal_attn_mask_xl(total_length, id_length, sa32, sa64, height, width, device=device, dtype=torch.float16)
    gradio_utils.total_count = sum(1 for name in unet.attn_processors.keys() if name.startswith('up_blocks') and not name.endswith('attn1.processor'))
    gradio_utils.attn_count = 0
    gradio_utils.id_length = id_length
    gradio_utils.total_length = total_length
    gradio_utils.cur_step = 0

inject_story_diffusion(pipe, num_prompts=len(script['panels']))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import sd_utils.gradio_utils as gradio_utils

def generate_comic(prompts):
    print(f'🎨 Đang vẽ {len(prompts)} khung tranh...')
    
    gradio_utils.attn_count = 0
    gradio_utils.cur_step = 0

    # Generate images as a batch to utilize StoryDiffusion's attention sharing
    generator = torch.Generator(device=device).manual_seed(42)
    out = pipe(prompt=prompts, height=768, width=768, num_inference_steps=25, generator=generator, guidance_scale=5.0)
    
    return out.images

images = generate_comic(script['panels'])

# Display results in a grid
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax, img, p in zip(axes.flatten(), images, script['panels']):
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(p[:40] + '...', fontsize=9)
plt.tight_layout()
plt.show()